# 第八课：RNN 循环神经网络

## 为什么需要 RNN？

CNN 处理的是**固定大小**的输入（如图像），但很多数据是**序列**：
- 文本：词的序列
- 语音：音频帧序列
- 时间序列：股票价格、天气数据

RNN 的核心思想：**共享参数，逐步处理序列，维护隐藏状态**

```
x1 → [RNN] → h1 → [RNN] → h2 → [RNN] → h3 → ...
         ↑              ↑              ↑
        h0             h1             h2
```

每个时间步使用相同的权重，隐藏状态 h 携带历史信息。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. RNN 基本原理

In [ ]:
# 手动实现一个简单 RNN
torch.manual_seed(42)

input_size = 3
hidden_size = 4

W_ih = torch.randn(hidden_size, input_size) * 0.01   # 输入到隐藏
W_hh = torch.randn(hidden_size, hidden_size) * 0.01  # 隐藏到隐藏
b_h = torch.zeros(hidden_size)

# 序列输入: 5个时间步，每步3维特征
seq_len = 5
x_seq = torch.randn(seq_len, input_size)
h = torch.zeros(hidden_size)  # 初始隐藏状态

print("手动 RNN 前向传播:")
print(f"初始隐藏状态: {h.numpy()}")

for t in range(seq_len):
    h = torch.tanh(x_seq[t] @ W_ih.T + h @ W_hh.T + b_h)
    print(f"  时间步 {t}: h = {h.detach().numpy().round(3)}")

print("\n公式: h_t = tanh(x_t @ W_ih^T + h_{t-1} @ W_hh^T + b_h)")

In [ ]:
# 用 PyTorch 的 nn.RNN
rnn = nn.RNN(input_size=3, hidden_size=4, batch_first=True)

x = torch.randn(1, 5, 3)   # (batch, seq_len, input_size)
h0 = torch.zeros(1, 1, 4)  # (num_layers, batch, hidden_size)

output, hn = rnn(x, h0)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")   # (1, 5, 4) - 每个时间步的隐藏状态
print(f"最终隐藏状态: {hn.shape}")    # (1, 1, 4) - 最后一个时间步
print()
print("output 包含所有时间步的隐藏状态")
print("hn 是最后一个时间步的隐藏状态")
print("output[:, -1, :] == hn[0] 应该相等")
print(f"验证: {torch.allclose(output[:, -1, :], hn[0])}")

In [ ]:
# nn.RNN 关键参数
print("=== nn.RNN 参数 ===")
print()
print("input_size:   输入特征维度")
print("hidden_size:  隐藏状态维度")
print("num_layers:   堆叠层数（默认1）")
print("batch_first:  True=输入(batch,seq,feature), False=输入(seq,batch,feature)")
print("bidirectional: True=双向RNN")
print()
print("输入输出形状 (batch_first=True):")
print("  输入:  (batch, seq_len, input_size)")
print("  输出:  (batch, seq_len, hidden_size)")
print("  h_n:   (num_layers, batch, hidden_size)")

## 2. RNN 的问题：梯度消失与梯度爆炸

In [ ]:
# 梯度消失演示
print("=== RNN 梯度问题 ===")
print()
print("RNN 的梯度需要通过时间步反向传播（BPTT）")
print("每个时间步乘以 W_hh，经过 T 步：梯度包含 (W_hh)^T")
print()
print("如果 W_hh 的最大特征值 > 1: 梯度指数增长 → 梯度爆炸")
print("如果 W_hh 的最大特征值 < 1: 梯度指数衰减 → 梯度消失")
print()

W = torch.tensor([[0.9, 0.0], [0.0, 0.9]])  # 特征值 0.9
grad = torch.tensor([1.0, 1.0])

print("梯度随时间步的变化 (特征值=0.9):")
for t in range(20):
    grad = grad @ W.T
    if t % 4 == 0:
        print(f"  步骤 {t+1:2d}: 梯度范数 = {grad.norm().item():.6f}")

print()
print("解决方案:")
print("  梯度裁剪 (gradient clipping) → 解决梯度爆炸")
print("  LSTM / GRU → 解决梯度消失")

In [ ]:
# 梯度裁剪
rnn = nn.RNN(input_size=10, hidden_size=20, batch_first=True)
x = torch.randn(4, 15, 10)
y = torch.randint(0, 2, (4,))

output, _ = rnn(x)
loss = output.sum()
loss.backward()

total_norm_before = sum(p.grad.norm().item() ** 2 for p in rnn.parameters() if p.grad is not None) ** 0.5
print(f"裁剪前梯度范数: {total_norm_before:.4f}")

torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)

total_norm_after = sum(p.grad.norm().item() ** 2 for p in rnn.parameters() if p.grad is not None) ** 0.5
print(f"裁剪后梯度范数: {total_norm_after:.4f}")
print()
print("torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)")
print("在 loss.backward() 之后、optimizer.step() 之前调用")

## 3. LSTM：长短期记忆网络

LSTM 通过门控机制解决 RNN 的梯度消失问题。

### 三个门

| 门 | 作用 |
|------|------|
| 遗忘门 f | 决定丢弃哪些旧信息 |
| 输入门 i | 决定存入哪些新信息 |
| 输出门 o | 决定输出哪些信息 |

### 核心公式
```
f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)   # 遗忘门
i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)   # 输入门
g_t = tanh(W_g @ [h_{t-1}, x_t] + b_g)      # 候选值
o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)   # 输出门

C_t = f_t * C_{t-1} + i_t * g_t              # 更新细胞状态
h_t = o_t * tanh(C_t)                        # 输出隐藏状态
```

关键：细胞状态 C_t 通过加法更新（而非乘法），梯度可以直接流过！

In [ ]:
# nn.LSTM 用法
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2, batch_first=True, dropout=0.1)

x = torch.randn(4, 15, 10)        # (batch, seq_len, input_size)
h0 = torch.zeros(2, 4, 20)        # (num_layers, batch, hidden_size)
c0 = torch.zeros(2, 4, 20)        # (num_layers, batch, hidden_size) - 细胞状态

output, (hn, cn) = lstm(x, (h0, c0))

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")      # (4, 15, 20)
print(f"隐藏状态 hn: {hn.shape}")   # (2, 4, 20)
print(f"细胞状态 cn: {cn.shape}")   # (2, 4, 20)
print()
print("LSTM 比 RNN 多一个细胞状态 c，用于长期记忆")
print("LSTM 参数量约为 RNN 的 4 倍（4个门）")

## 4. GRU：门控循环单元

GRU 是 LSTM 的简化版，合并了细胞状态和隐藏状态，只有两个门。

```
z_t = sigmoid(W_z @ [h_{t-1}, x_t])   # 重置门
r_t = sigmoid(W_r @ [h_{t-1}, x_t])   # 更新门
h_tilde = tanh(W_h @ [r_t * h_{t-1}, x_t])  # 候选值
h_t = (1 - z_t) * h_{t-1} + z_t * h_tilde   # 更新隐藏状态
```

GRU 参数更少，训练更快，很多任务上效果与 LSTM 相当。

In [ ]:
# nn.GRU 用法
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=2, batch_first=True)

x = torch.randn(4, 15, 10)
h0 = torch.zeros(2, 4, 20)

output, hn = gru(x, h0)

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")
print(f"隐藏状态: {hn.shape}")
print()
print("GRU 和 RNN 接口完全一致，只是内部计算不同")
print("GRU 比 LSTM 参数少（2个门 vs 4个门），但效果通常差不多")

In [ ]:
# RNN vs LSTM vs GRU 参数量对比
input_size = 10
hidden_size = 20

rnn = nn.RNN(input_size, hidden_size, batch_first=True)
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
gru = nn.GRU(input_size, hidden_size, batch_first=True)

rnn_params = sum(p.numel() for p in rnn.parameters())
lstm_params = sum(p.numel() for p in lstm.parameters())
gru_params = sum(p.numel() for p in gru.parameters())

print(f"RNN  参数量: {rnn_params:,}")
print(f"LSTM 参数量: {lstm_params:,} ({lstm_params/rnn_params:.1f}x RNN)")
print(f"GRU  参数量: {gru_params:,} ({gru_params/rnn_params:.1f}x RNN)")
print()
print("LSTM 参数 ≈ 4 * RNN 参数（4个门）")
print("GRU  参数 ≈ 3 * RNN 参数（2个门 + 候选值）")

---
## 5. 实战：正弦波预测

用 RNN 预测正弦波的未来值。

In [ ]:
# 生成正弦波数据
torch.manual_seed(42)

T = 200
t = torch.linspace(0, 20, T)
data = torch.sin(t) + 0.1 * torch.randn(T)

plt.figure(figsize=(12, 3))
plt.plot(t.numpy(), data.numpy())
plt.xlabel('Time')
plt.ylabel('Value')
plt.title('Sine Wave with Noise')
plt.grid(True)
plt.show()

# 创建序列数据
seq_length = 20

def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return torch.stack(X), torch.stack(y)

X_seq, y_seq = create_sequences(data, seq_length)
X_seq = X_seq.unsqueeze(-1)  # (batch, seq_len, 1)
y_seq = y_seq.unsqueeze(-1)  # (batch, 1)

print(f"X shape: {X_seq.shape}")  # (180, 20, 1)
print(f"y shape: {y_seq.shape}")  # (180, 1)

In [ ]:
# 定义 RNN 预测模型
class SequencePredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, rnn_type='lstm'):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        if rnn_type == 'rnn':
            self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        elif rnn_type == 'lstm':
            self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        elif rnn_type == 'gru':
            self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # 只取最后一个时间步
        return out

model = SequencePredictor(rnn_type='lstm')
print(model)

In [ ]:
# 训练
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

train_size = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

losses = []
for epoch in range(100):
    model.train()
    pred = model(X_train)
    loss = criterion(pred, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}: loss={loss.item():.6f}")

print(f"\n最终训练 loss: {losses[-1]:.6f}")

In [ ]:
# 可视化预测结果
model.eval()
with torch.no_grad():
    train_pred = model(X_train).numpy()
    test_pred = model(X_test).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True)

axes[1].plot(y_train.numpy(), label='True', alpha=0.7)
axes[1].plot(train_pred, label='Predicted', alpha=0.7)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Value')
axes[1].set_title('Sine Wave Prediction')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6. 对比 RNN / LSTM / GRU

In [ ]:
rnn_types = ['rnn', 'lstm', 'gru']
results = {}

for rnn_type in rnn_types:
    torch.manual_seed(42)
    model = SequencePredictor(hidden_size=32, rnn_type=rnn_type)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    type_losses = []
    for epoch in range(100):
        model.train()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        type_losses.append(loss.item())
    
    model.eval()
    with torch.no_grad():
        test_loss = criterion(model(X_test), y_test).item()
    
    results[rnn_type.upper()] = type_losses
    print(f"{rnn_type.upper():5s}: 训练 loss={type_losses[-1]:.6f}, 测试 loss={test_loss:.6f}")

plt.figure(figsize=(8, 5))
for name, l in results.items():
    plt.plot(l, label=name, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RNN vs LSTM vs GRU')
plt.legend()
plt.grid(True)
plt.show()

print("LSTM 和 GRU 通常比普通 RNN 收敛更快、效果更好")

---
## 7. 双向 RNN 与多层 RNN

In [ ]:
# 双向 LSTM
bilstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2,
                  batch_first=True, bidirectional=True)

x = torch.randn(4, 15, 10)
output, (hn, cn) = bilstm(x)

print(f"输入: {x.shape}")
print(f"输出: {output.shape}")  # (4, 15, 40) - 双向所以 hidden_size*2
print(f"hn: {hn.shape}")        # (4, 4, 20) - num_layers*2
print()
print("双向 RNN 同时看前向和后向信息")
print("输出维度 = hidden_size * 2")
print("适用于: 文本分类、命名实体识别等需要完整上下文的任务")
print("不适用于: 实时预测（后向信息来自未来）")

In [ ]:
# 多层 RNN
multi_lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=3,
                      batch_first=True, dropout=0.2)

x = torch.randn(4, 15, 10)
output, (hn, cn) = multi_lstm(x)

print(f"3层 LSTM:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  hn: {hn.shape}")  # (3, 4, 20)
print()
print("num_layers: 堆叠的 RNN 层数")
print("dropout: 层间的 dropout（仅中间层，最后一层不加）")
print("通常 2-3 层就够了，太深容易过拟合")

## 8. 文本分类实战（简化版）

用 LSTM 对文本情感进行分类。

In [ ]:
# 模拟文本数据
torch.manual_seed(42)

vocab_size = 1000
embed_dim = 32
hidden_dim = 64
num_classes = 2
n_samples = 500
max_len = 30

# 随机生成文本序列（词ID）
X_text = torch.randint(0, vocab_size, (n_samples, max_len))
y_text = torch.randint(0, num_classes, (n_samples,))

print(f"文本数据: X={X_text.shape}, y={y_text.shape}")
print(f"词ID范围: 0-{vocab_size-1}")
print(f"序列长度: {max_len}")

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # 词嵌入
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # 双向所以 *2
    
    def forward(self, x):
        # x: (batch, seq_len) 词ID
        embedded = self.embedding(x)       # (batch, seq_len, embed_dim)
        output, _ = self.lstm(embedded)    # (batch, seq_len, hidden_dim*2)
        out = output[:, -1, :]             # 取最后时间步 (batch, hidden_dim*2)
        logits = self.fc(out)              # (batch, num_classes)
        return logits

text_model = TextClassifier(vocab_size, embed_dim, hidden_dim, num_classes)
print(text_model)
print(f"\n参数量: {sum(p.numel() for p in text_model.parameters()):,}")

In [ ]:
# 训练
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(text_model.parameters(), lr=0.001)

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_text, y_text)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

for epoch in range(20):
    text_model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in loader:
        logits = text_model(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(text_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}: loss={total_loss/total:.4f}, acc={correct/total:.4f}")

print("\n文本分类模型训练完成")
print("关键组件: Embedding -> BiLSTM -> Linear")

## 9. nn.Embedding：词嵌入

In [ ]:
# Embedding 详解
embedding = nn.Embedding(num_embeddings=1000, embedding_dim=32)

# 输入是词的 ID
word_ids = torch.tensor([0, 5, 42, 999])
vectors = embedding(word_ids)

print(f"词ID: {word_ids.tolist()}")
print(f"嵌入向量形状: {vectors.shape}")  # (4, 32)
print(f"每个词被映射为 {vectors.shape[1]} 维的向量")
print()
print("Embedding 本质: 一个查找表")
print("  输入: 词的整数 ID")
print("  输出: 对应的嵌入向量")
print("  参数量: vocab_size * embedding_dim")
print(f"  本例参数量: {1000 * 32:,}")
print()
print("嵌入向量在训练中学习，语义相近的词向量也相近")

---
## 总结

### RNN 家族对比

| 模型 | 门控 | 参数量 | 优点 | 缺点 |
|------|------|--------|------|------|
| RNN | 无 | 1x | 简单 | 梯度消失 |
| LSTM | 3个门 | 4x | 长程记忆 | 参数多 |
| GRU | 2个门 | 3x | 轻量高效 | 表达力略弱 |

### RNN 常见架构

```
1. 序列分类: Embedding -> RNN -> 取最后时间步 -> Linear
2. 序列标注: Embedding -> BiRNN -> 每个时间步 -> Linear
3. 序列到序列: Encoder(Embedding -> RNN) -> Decoder(RNN -> Linear)
```

### 关键实践
- 优先使用 LSTM 或 GRU，而非普通 RNN
- 训练时使用**梯度裁剪**
- batch_first=True 更直观
- 双向 RNN 适用于离线任务
- Embedding 层将离散 ID 映射为连续向量

### RNN vs Transformer
```
RNN:  串行处理，长序列慢，但参数少
Transformer: 并行处理，长序列快，但参数多
目前 Transformer 已成为主流，但 RNN 在小数据和实时场景仍有价值
```